# Final test-set evaluation (week 4)
Scores the selected GRPO checkpoint (LoRA adapter attached as a dataset) on GPU 0 and re-scores the base model on GPU 1, with the week 1 harness settings: vLLM, fp16, greedy, 1024 tokens. Run once; test sets are not used for any choice.

In [ ]:
import os, subprocess
REPO, SRC = "https://github.com/Vansh8010-boop/grpo-reasoning.git", "/tmp/grpo-reasoning"
if os.path.isdir(SRC):
    subprocess.run(["git", "-C", SRC, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm "math-verify[antlr4_13_2]" 2>&1 | tail -3
# vLLM brings its own torch build; Kaggle's preinstalled torchaudio no longer matches it
# and transformers imports it on startup, so remove it (no audio needed here).
!pip uninstall -y -q torchaudio
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__)"

In [ ]:
!python -m pytest -q tests/ 2>&1 | tail -3

In [ ]:
import glob
OUT = "/kaggle/working"
adapters = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)]
assert len(adapters) == 1, adapters
print("adapter:", adapters[0])
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
jobs = {"grpo-1.5b-step350": (BASE, "0", ["--adapter", adapters[0]]), "base-1.5b-rerun": (BASE, "1", [])}
env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
procs = {}
for name, (model, gpu, extra) in jobs.items():
    cmd = ["python", "-m", "eval.harness", "--model", model, "--dtype", "float16", *extra, "--out", f"{OUT}/results/{name}"]
    procs[name] = subprocess.Popen(cmd, env={**env, "CUDA_VISIBLE_DEVICES": gpu},
                                   stdout=open(f"{OUT}/{name}.log", "w"), stderr=subprocess.STDOUT)
for name, p in procs.items():
    print(name, "exit code", p.wait())
for name in jobs:
    print(f"
===== {name} =====")
    print("".join(l for l in open(f"{OUT}/{name}.log") if l.startswith("[") or "Error" in l)[-3000:])

In [ ]:
import json
rows = []
for name in jobs:
    path = f"{OUT}/results/{name}/summary.json"
    if not os.path.exists(path):
        print("missing", path); continue
    for ds, s in json.load(open(path))["datasets"].items():
        rows.append(f"{name:10s} {ds:8s} acc {s['accuracy']*100:5.1f}%  boxed {s['boxed_rate']*100:5.1f}%  "
                    f"trunc {s['truncated_rate']*100:4.1f}%  tokens {s['avg_tokens']:4.0f}  {s['seconds']}s")
print("\n".join(rows))
os.makedirs(f"{OUT}/failures", exist_ok=True)
for name in jobs:
    for ds in ("gsm8k", "math500", "svamp"):
        src = f"{OUT}/results/{name}/{ds}.jsonl"
        if os.path.exists(src):
            subprocess.run(["python", "-m", "eval.failures", src, "-n", "20"],
                           stdout=open(f"{OUT}/failures/{name}-{ds}.md", "w"), check=True)